# 05b — Selecting k by stability

Pointwise quality metrics (silhouette, Calinski-Harabasz) increase monotonically with k and do not allow a natural k to be fixed. Here k is chosen by stability: the clustering is repeated over 80% subsamples of the users and both the mean silhouette and the ARI between repetitions are measured. The autoencoder space is reconstructed from the files saved by notebook 05.

## 0 · Libraries and paths

In [ ]:
import pickle
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Add src/ to the path and reuse the shared helper (same pattern as 01-04)
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import find_project_root  # noqa: E402

PROCESSED_PATH = find_project_root() / "data" / "processed"
np.random.seed(42)
print("Carpeta de datos:", PROCESSED_PATH)

## 1 · Reconstructing the autoencoder space (X_ae)

The transformations from notebook 05 (variable encoding, scaling and the pass through the autoencoder) are repeated using the saved files, in order to work on the same space.

In [ ]:
users = pd.read_csv(PROCESSED_PATH / "users.csv", dtype={"cp_num": str})

d = users.copy()
d["gender_enc"] = (d["gender"] == "H").astype(int)
d["labor_status_enc"] = d["labor_status"].map({"employed": 2, "unemployed": 1, "inactive": 0}).fillna(0).astype(int)
d["civil_status_enc"] = d["civil_status"].map({"casado": 3, "soltero": 0, "divorciado": 1, "viudo": 2}).fillna(0).astype(int)
d["tiene_coche_enc"] = d["tiene_coche"].astype(int)
d["num_room_enc"] = d["num_room"].map({"menos_3_hab": 1, "3_a_6_hab": 2, "7_mas_hab": 3}).fillna(2).astype(int)


def tamano_hogar(s):
    # SAME encoding as parse_hogar() in notebook 05 (size_hogar runs from 1 to 5 persons),
    # so as to validate exactly the same space as there.
    if pd.isna(s): return 3
    s = str(s).strip()
    if s.startswith("1"):   return 1
    elif s.startswith("2"): return 2
    elif s.startswith("3"): return 3
    elif s.startswith("4"): return 4
    else:                   return 5


d["size_hogar_enc"] = d["size_hogar"].apply(tamano_hogar)

def edad_a_grupo(edad):
    # Age bands (the same as model M2): 18-24, 25-34, 35-44, 45-54, 55-64, 65+
    if pd.isna(edad): return -1
    for i, lim in enumerate([25, 35, 45, 55, 65]):
        if edad < lim: return i
    return 5
d["age_cat"] = d["age"].apply(edad_a_grupo)

# Age is BANDED into categories (age_cat), as in 05 and in M2
DEMO_FEATS = ["age_cat", "gender_enc", "labor_status_enc", "civil_status_enc", "tiene_coche_enc",
              "size_hogar_enc", "num_room_enc", "ipa_class", "mun_type", "distance_type"]
X_raw = d[DEMO_FEATS].fillna(-1).values

# Load the scaler and (if used) the autoencoder saved by 05
with open(PROCESSED_PATH / "demo_scaler.pkl", "rb") as f:
    escalador = pickle.load(f)
with open(PROCESSED_PATH / "demo_ae_used.pkl", "rb") as f:
    se_uso_autoencoder = pickle.load(f)

X_escalada = escalador.transform(X_raw)
if se_uso_autoencoder:
    import tensorflow as tf
    encoder = tf.keras.models.load_model(str(PROCESSED_PATH / "demo_encoder.keras"))
    X_ae = encoder.predict(X_escalada, verbose=0)
else:
    X_ae = X_escalada

print("Espacio de clustering X_ae:", X_ae.shape, "| autoencoder usado:", se_uso_autoencoder)

## 2 · Evaluating each k by resampling

For each k from 2 to 12 the clustering is repeated 15 times over 80% of the users and the silhouette of each subsample is recorded. The ARI is computed between each pair of the 15 groupings, projected onto all users.

In [ ]:
K_RANGE = range(2, 13)
N_REPETICIONES = 15
FRACCION = 0.8

n_usuarios = len(X_ae)
rng = np.random.RandomState(42)
filas = []

for k in K_RANGE:
    silhouettes = []
    agrupaciones = []   # the cluster label of ALL users, in each repetition

    for repeticion in range(N_REPETICIONES):
        # 1) take 80% of the users at random
        indices = rng.choice(n_usuarios, int(n_usuarios * FRACCION), replace=False)

        # 2) KMeans on that slice
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        km.fit(X_ae[indices])

        # silhouette of that slice
        sil = silhouette_score(X_ae[indices], km.labels_, sample_size=5000, random_state=42)
        silhouettes.append(sil)

        # cluster label of ALL users (to compare between repetitions)
        agrupaciones.append(km.predict(X_ae))

    # 3) ARI between each pair of repetitions
    aris = []
    for i in range(N_REPETICIONES):
        for j in range(i + 1, N_REPETICIONES):
            aris.append(adjusted_rand_score(agrupaciones[i], agrupaciones[j]))

    silhouettes = np.array(silhouettes)
    aris = np.array(aris)
    filas.append({
        "k": k,
        "sil_media": silhouettes.mean(),
        "sil_lo": np.percentile(silhouettes, 2.5),
        "sil_hi": np.percentile(silhouettes, 97.5),
        "ari_media": aris.mean(),
        "ari_std": aris.std(),
    })

resultados = pd.DataFrame(filas)
print(resultados.round(4).to_string(index=False))

In [ ]:
# Plots: silhouette (with its interval) and ARI by k
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(resultados["k"], resultados["sil_media"], "o-", color="steelblue")
axes[0].fill_between(resultados["k"], resultados["sil_lo"], resultados["sil_hi"], alpha=0.2, color="steelblue")
axes[0].axvline(10, color="crimson", linestyle="-", label="k=10 (adoptado)")
axes[0].axvline(12, color="gray", linestyle=":", label="k=12 (máx silueta)")
axes[0].set_title("Silhouette (media e intervalo)")
axes[0].set_xlabel("k")
axes[0].legend()

axes[1].errorbar(resultados["k"], resultados["ari_media"], yerr=resultados["ari_std"],
                 fmt="o-", color="darkorange", capsize=3)
axes[1].axhline(0.9, color="gray", linestyle="--", linewidth=1, label="ARI=0.90 (estable)")
axes[1].axvline(10, color="crimson", linestyle="-")
axes[1].axvline(12, color="gray", linestyle=":")
axes[1].set_title("Estabilidad (ARI entre repeticiones)")
axes[1].set_xlabel("k")
axes[1].legend()

plt.tight_layout()
plt.show()

k_mejor_sil = int(resultados.loc[resultados["sil_media"].idxmax(), "k"])
k_mejor_ari = int(resultados.loc[resultados["ari_media"].idxmax(), "k"])
ari_en_10 = float(resultados.loc[resultados["k"] == 10, "ari_media"].iloc[0])
print("k con mayor silhouette :", k_mejor_sil)
print("k más estable (ARI)    :", k_mejor_ari)
print("ARI en k=10            :", round(ari_en_10, 4))

## Conclusion

- The **silhouette increases monotonically** with k (from 0.35 at k=2 to 0.46 at k=12): the demographic space
  behaves as a **continuum**, with no natural number of groups. For this reason its maximum (k=12) coincides with
  the upper end of the search range and **is not a reliable criterion**.
- The decisive criterion is **stability (ARI)**. There are local maxima of stability at k=2 (0.999,
  trivial), k=6 (0.957) and **k=10 (0.977)**, the latter clearly above its neighbours (k=9 ≈ 0.88;
  k=11 ≈ 0.94).
- **k=10 is adopted**: it is the most cohesive clustering (silhouette ≈ 0.43) that is also **stable** (ARI
  ≈ 0.98) and lies in the **interior** of the range (not at the boundary). k=12 has slightly better metrics but is
  the upper end of the search (risk of an artefact); k=6 is more parsimonious but less cohesive.

This k=10 propagates to the rest of the pipeline (M2 propensity, M3 cold start and the inverse model).